In [ ]:
import pandas as pd

# 1. Import des données

In [ ]:
df_irve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'
)

print(f"Shape : {df_irve.shape}")
df_irve.sample(5)

Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).
Cette table comporte 212234 lignes pour 52 variables.

In [ ]:
df_ve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
 encoding='latin-1'
)

print(f"Shape : {df_ve.shape}")
df_ve.sample(5)

Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge. Il contient l'historique des immatriculations pour chaque commune à différentes dates.
Cette table comporte 703545 lignes pour 8 variables. 

In [ ]:
df_revenus = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',sep=';'
)

print(f"Shape : {df_revenus.shape}")
df_revenus.sample(5)

Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 34926 lignes pour 57 variables.

# 2. Jointure des tables

## 2.1 Choix de la variable de jointure

Les 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus

La jointure sera faite sur ces variables.

## 2.2 Préparation de la variable de jointure

Le diagnostic met en évidence plusieurs points d’attention :
- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

### 2.2.1 Unification des codes

In [ ]:
from src.preparation_data import nettoyer_code_insee

In [ ]:
df_irve["code_insee_commune"] = df_irve["code_insee_commune"].apply(nettoyer_code_insee)
df_ve["CODGEO"] = df_ve["CODGEO"].apply(nettoyer_code_insee)
df_revenus["Code géographique"] = df_revenus["Code géographique"].apply(nettoyer_code_insee)

Les codes insee sont maintenant au même format dans les 3 bases de données : un code à 5 caractères.

### 2.2.2 Compléter les valeurs manquantes

In [ ]:
from src.preparation_data import creer_gdf_irve, charger_communes, joindre_communes, ajouter_codes_geo

In [ ]:
gdf_irve = creer_gdf_irve(df_irve, "consolidated_longitude", "consolidated_latitude")
gdf_communes = charger_communes()
gdf_result = joindre_communes(gdf_irve, gdf_communes)
df_irve = ajouter_codes_geo(df_irve, gdf_result)

Nous avons maintenant 2 colonnes concernant le code géographique dans la base de données IRVE :
- `code_insee_commune` : la variable initiale
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude

In [ ]:
from src.preparation_data import compter_valeurs_manquantes, compter_uniques

In [ ]:
colonnes_irve = ["code_insee_commune", "code_geo_total"]

print("Valeurs manquantes :")
print(compter_valeurs_manquantes(df_irve, colonnes_irve))

print("\nValeurs uniques :")
print(compter_uniques(df_irve, colonnes_irve))

Cette manipulation permet de passer de 57919 valeurs manquantes à 1768. De plus les codes recalculés sont plus fiables que ceux initiaux. En effet, les données entrées n'étant pas toujours vérifiées, des erreurs étaient présentes. Certaines code géographiques correspondaient au code postal par exemple.

## 2.3 Jointure

Réfléchissons aux variables à garder... Lesquelles pourraient être intéressante ?

In [ ]:
list(df_irve.columns)

In [ ]:
list(df_ve.columns)

In [ ]:
list(df_revenus.columns)

### 2.3.1 Une ligne par code géo

Pour pouvoir efectuer la jointure sur les codes géographiques, il est essentiel que pour chacune de nos 3 bases de données, chaque ligne corresponde à un unique code géographique.
Il faut alors réfléchir à la façon dont on veut agréger df_irve et quelles variables on souhaite garder.
Il faut également étudier la base df_ve afin de supprimer les "doublons" de code géographique.

##### df_irve

Ici, l'objectif est de mesurer l'offre de recharge par commune. Comme il peut y avoir plusieurs points de recharge par commune, il faut agréger. L'objectif final est d'expliquer ou prédire le taux de véhicules électriques local.

Après étude des nos variables (cf etude_df_irve.ipynb) nous avons sélectionner plusieurs variables nous permettant d'en créer de nouvelles, agrégées par code géographique.

Certaines variables ont été écarté directement car jugées non pertinente pour notre sujet, d'autres présetaient trop de valeurs manquantes, d'autres étaient trop peu informatives ou encore était du texte de libre donc trop difficile à utiliser avec le peu de temps dont nous disposons.

variables utilisées (12 variables) pour la création des variables agrégées :`code_geo_total`,`nom_operateur`,
               'implantation_station',
               'nbre_pdc',
               'puissance_nominale',
               'prise_type_ef',
               'prise_type_2',
               'prise_type_combo_ccs',
               'prise_type_chademo',
               'gratuit',
               'paiement_cb',
               'paiement_autre'

variables créées : une ligne finale = un code géographique.

|variables utilisées| Variable finale   | Construction      |
|| ----------------- | ----------------- |
|| total_pdc         | somme             |
|| puissance_moyenne | moyenne           |
|| puissance_max     | max               |
|| nb_operateurs     | nunique           |
|| top_operateur     | mode              |
|| pct_type_2        | moyenne booléenne |
|| pct_gratuit       | moyenne booléenne |
|| part_voirie       | dummies + moyenne |

Compléter uniquement se tableau très proprement

Commençons par rendre les données des variables sélectionner propres avant de faire l'agrégation.

In [ ]:
from src.nouvelles_variables import clean_irve_variables_finales

df_filtre = clean_irve_variables_finales(df_irve)
df_filtre.sample(5)

Nous pouvons maintenant procéder à la création des variables agrégées.

In [ ]:
from src.nouvelles_variables import creer_features_irve

nv_var = creer_features_irve(df_filtre)
nv_var.sample(5)

##### df_ve

La base véhicules contient plusieurs dates d'observation pour une même commune.

Afin d'être cohérent avec les autres sources de données (IRVE, INSEE), nous souhaitons disposer d'une seule ligne par commune correspondant à la situation la plus récente disponible.

Nous conservons donc, pour chaque code commune (`CODGEO`), la dernière observation temporelle.

In [ ]:
from src.nouvelles_variables import preparer_base_ve

In [ ]:
df_ve_latest = preparer_base_ve(df_ve)
print(df_ve_latest.shape)
df_ve_latest.head()

Deux variables sont conservées :

- `NB_VP` : nombre total de voitures particulières
- `NB_VP_RECHARGEABLES_EL` : nombre de voitures rechargeables / électriques

Plutôt que d'utiliser le nombre brut de véhicules électriques, nous construisons un ratio :

\[
taux\_equipement\_ve = \frac{NB\_VP\_RECHARGEABLES\_EL}{NB\_VP}
\]

Ce ratio est plus pertinent car il neutralise l'effet de taille des communes.

In [ ]:
df_ve_latest["taux_equipement_ve"].describe().round(4)

##### df_revenus

sous-partie à faire

In [ ]:
# On garde le code commune et le revenu médian (souvent colonne 'MED21' pour 2021)
df_revenus_final = df_revenus[['COM', 'MED21']].copy()
df_revenus_final.rename(columns={'COM': 'code_commune_insee', 'MED21': 'revenu_median'}, inplace=True)

**Variables à garder :**
[DISP] Médiane (€), [DISP] Part des revenus d’activité (%), [DISP] Iice de Gini.

Intérêt : La médiane du revenu est indispensable pour vérifier si l'adoption du VE est corrélée à la richesse de la commune. L'indice de Gini peut montrer si les communes très inégalitaires ont un comportement différent.

On peut en choisir d'autres. D'ailleurs on doit choisir uniquement les variables de df_irve avant la jointure car pour les 2 autres tables on a déjà une ligne par commune donc pas besoin de faire d'agrégation. Ainsi on peut toutes les prendre et choisir après.

In [ ]:
# jointure
df_final = (
    df_ve_latest
    .merge(nv_var, left_on="CODGEO", right_on="code_geo_total", how="left")
    .merge(df_revenus, left_on="CODGEO", right_on="Code géographique", how="left")
)

print("Shape base finale :", df_final.shape)

df_final.head()

# 3. Analyse des variables

Après le choix des variables : 
0. variable cible = taux de VE
1. faire des analyses descriptives : 
- automatiser avec une fonction
- matrice de corrélation (heatmap)
2. modélisation :
simple regression ou plus avace (random forest...)

# 4. Modélisation

In [ ]:
from src.modelisation import modelisation_lasso, modelisation_xgboost

Nos variables pour la modélisation sont les suivantes :

attention LASSO ne prend pas de valeurs manquantes !!

In [ ]:
features = ['total_pdc', 'puissance_max', 'nb_operateurs', 'pct_gratuit']
target = 'taux_equipement_ve' 

df_model = df_final[features + [target]].dropna()

# 2. Exécuter le Lasso (Baseline & Sélection)
model_l, metrics_l, coeffs_l = modelisation_lasso(df_model, features, target)
print(f"R2 Lasso : {metrics_l['r2']:.3f}")
print("Déterminants (Lasso) :\n", coeffs_l[coeffs_l != 0])

# 3. Exécuter le Gradient Boosting (Performance)
model_x, metrics_x, importance_x = modelisation_xgboost(df_final, features, target)
print(f"R2 XGBoost : {metrics_x['r2']:.3f}")
importance_x.plot(kind='barh', title="Importance des variables - XGBoost")